# Parte V: Segmentación guiada por texto con SAM

## Setup previo

Para poder ejecutar esta prate del seminario correctamente necesitaremos tener una GPU. Asegurate de que tu runtime de Colab este usando GPU tal siguiendo las indicaciones de las siguientes imagenes:

<br>
<div style="display: flex; gap: 16px; justify-content: center; flex-wrap: wrap;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/RunTime_Colab.png" width="400">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/GPU_Selecton_Colab.png" width="400">
</div>

## Fundamento teórico

**SAM** (Segment Anything Model, Meta 2023) es un modelo que está entrenado para poder segmentar cualquier objeto de una imagen generando automáticamente todas las máscaras posibles, sin saber qué son. Esta segmentación es puramente visual, a nivel de píxeles. No contiene ningún tipo de información semantica. No obstante, podemos reaprovechar su *output* para poder pasar esas mascaras de manera individual por el encoder visual y obtener un embedding de ese trozo de imagen de manera aislada. Esta solución nos permitirá comparar cada una de las máscaras con prompts de texto, generando así un *buscador de objetos* para una imagen.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/SAM_Example.png" width="600">
</div>

Combinado con CLIP:
1. SAM genera todas las máscaras posibles
2. Para cada máscara: recortamos la región y la codificamos con CLIP
3. Comparamos con el embedding del texto query
4. La máscara con mayor similitud = el objeto buscado

> **Debate:** ¿Para qué aplicaciones reales os parece útil la segmentación guiada por texto?

## Setup
Instalamos las librerías necesarias y descargamos los **pesos** de SAM (`vit_h`, ~2.5GB). Cuando se usa un modelo entrenado, normalmente hay que indicar que versión del modelo queremos usar. Suelen existir distintas versiones (más o menos pesadas, diferentes arquitecturas, diferentes imagenes de entrenamiento para diferenctes especializaciones, etc..) Los pesos contienen todo lo que el modelo aprendio durante el entrenamiento. Sin ellos la red es solo una arquitectura vacía.

Para SAM en este caso usaremos `sam_vit_h_4b8939.pth`.

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
!pip install open_clip_torch

In [ ]:
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

**Cargamos la imagen** que vamos a segmentar. Durante esta parte del seminario usaremos la GPU para poder procesar los datos más rapidamente. No obstante la capacidad que Colab nos deja usar para las GPUs es limitada. Una gran parte de la RAM de la GPU está ya ocupada por el modelo de SAM. Cuando carguemos la imagen en la GPU podriamos saturarla. Es por ello que como la imagen que usaremos esta en 4K vamos a hacerle un redimensionamiento (**downsampling**) a 1024x768, y la convertimos a **array numpy**, que es el formato que espera el modelo.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import requests
from io import BytesIO

def download_and_resize_image(url, size=(1024, 768)):
    """Descarga una imagen desde una URL, la convierte a RGB y la redimensiona."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB").resize(size)
        return img, np.array(img)
    except Exception as e:
        print(f"Error al cargar la imagen: {e}")
        return None, None

img_src_pth = 'https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Desktop_Segmentation.jpg'
img_src, img_src_np = download_and_resize_image(img_src_pth)

if img_src is not None:
    plt.imshow(img_src)
    plt.axis('off')
    plt.show()

Cargamos SAM en la GPU. Asegurate de que `Device used` es **CUDA** para estar trabajadno en la GPU.

Tambiñen ceamos `SamAutomaticMaskGenerator`. SAM esta pensado para hacer click en la iamagen y devolvernos la mascara del objeto que hemos clicado. Para esta práctica necesitamos tener todas las mascaras de la imagen. `SamAutomaticMaskGenerator` escanea toda la imagen y genera **todas las máscaras posibles** sin ninguna indicación.

In [ ]:
import torch

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device used: ", device)
sam = sam_model_registry['vit_h'](checkpoint='./sam_vit_h_4b8939.pth').to(device=device)
mask_generator = SamAutomaticMaskGenerator(sam)

Definimos dos funciones auxiliares. `show_anns` superpone las máscaras sobre la imagen con colores aleatorios. `convert_box_xywh_to_xyxy` convierte el formato del bounding box de SAM al que necesitamos para recortar.

A continuación generamos todas las máscaras (~1 min).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_anns(anns):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:,:,3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        img[m] = color_mask
    ax.imshow(img)

def convert_box_xywh_to_xyxy(box):
    x1 = box[0]
    y1 = box[1]
    x2 = box[0] + box[2]
    y2 = box[1] + box[3]
    return [x1, y1, x2, y2]

In [ ]:
sam_result_src = mask_generator.generate(img_src_np)
plt.figure(figsize=(10,10))
plt.imshow(img_src_np)
show_anns(sam_result_src)
plt.axis('off')
plt.show()

Para poder codificar cada región con CLIP necesitamos extraer cada máscara como imagen independiente. `segment_image_black_and_crop` pone a negro todo lo exterior a la máscara y recorta el **bounding box** (El rectangulo más pequeño que engloba todo el objeto de la mascara). Guardamos también todas las máscaras booleanas en `batch_src_binary_mask` para la visualización final.

In [ ]:
def segment_image_black_and_crop(img_np, segmentation_mask, bbox_xyxy):
    # Create a copy of the image and apply the mask
    # The mask is boolean, so we set everywhere else to 0 (black)
    black_image = img_np.copy()
    black_image[~segmentation_mask] = 0

    # Crop the image using the bounding box [x1, y1, x2, y2]
    x1, y1, x2, y2 = map(int, bbox_xyxy)
    black_image = black_image[y1:y2, x1:x2]

    return black_image

In [ ]:
def extract_crops_and_masks(img_np, sam_result):
    """Recorta cada máscara de SAM y devuelve (crops, binary_masks)."""
    crops = [
        segment_image_black_and_crop(img_np, m["segmentation"], convert_box_xywh_to_xyxy(m["bbox"]))
        for m in sam_result
    ]
    masks = np.zeros((len(sam_result), img_np.shape[0], img_np.shape[1])) > 0
    for k, m in enumerate(sam_result):
        masks[k] = m["segmentation"]
    return crops, masks

In [ ]:
cropped_boxes_black, batch_src_binary_mask = extract_crops_and_masks(img_src_np, sam_result_src)


## Visualización de los recortes
Visualizamos todos los recortes. Cada imagen es un objeto que SAM ha detectado. ¿Reconoces todos? ¿Hay alguno que no esperabas?

Recuerda que SAM no entiende de semántica. El ha detctado objetos y lso ha recoortado gracias a su red entrenada, pero no es capaz de entender que está recortando.

In [ ]:
def visualize_cropped_masks(cropped_boxes_black):
  num_images = len(cropped_boxes_black)
  cols = np.min([8,num_images])
  rows = (num_images + cols - 1) // cols  # Compute the number of rows needed
  fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
  axes = np.array(axes).reshape(rows, cols)  # Ensure it's a 2D array

  for i, ax in enumerate(axes.flatten()):
      if i < num_images:
          ax.imshow(cropped_boxes_black[i], cmap='gray')  # Adjust cmap if needed
          ax.axis('off')  # Hide axes
          ax.set_title(str(i))
      else:
          ax.axis('off')  # Hide empty subplots

  plt.tight_layout()
  plt.show()

visualize_cropped_masks(cropped_boxes_black)


## Mascaras de SAM + CLIP

Ahora que ya tenemos las mascaras de los objetos de las imagenes es cuando podemos empezar a jugar con CLIP y embeddings tanto de texto como visuales.

Para este caso usaremos un modelo diferente a **CLIP (OpenAI)**. Cargamos **SigLIP2**, el modelo multimodal de **Google** equivalente a CLIP pero más potente. Funciona exactamente como CLIP, solo cambia la forma de haber sido entrenado. Tiene un **encoder de texto** y **otro de imagen** que proyectan ambas modalidades al **mismo espacio vectorial**, de forma que la similitud coseno entre un texto y una imagen tiene sentido semántico. La descarga pesa ~4.5GB, tardará un par de minutos.

Tal y como hicimos con SAM, para CLIP también le indicaremos que pesos concretos queremos usar `ViT-SO400M-16-SigLIP2-384` y cargaremos el modelo en la GPU `Device used Cuda` para que la inferencia sea mucho más rápida.

In [ ]:
import torch
import open_clip
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device used: ", device)
model, _, preprocess = open_clip.create_model_and_transforms('ViT-SO400M-16-SigLIP2-384', pretrained='webli',device=device)
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer('ViT-SO400M-16-SigLIP2-384')

In [ ]:
def cosine_similarity(a, b):
    # Normaliza cada vector dividiendo por su norma (pista: .norm(dim=-1, keepdim=True))
    a = ... # <-- COMPLETE HERE
    b = ... # <-- COMPLETE HERE
    # El producto escalar de vectores normalizados ES la similitud coseno
    # Pista: En Python se usa el operador @ para multiplicación matricial
    # La segunda matriz ha de estar transpuesta (.T)
    # Devuelve el resultado en CPU (.cpu())
    return ... # <-- COMPLETE HERE

def encode_with_siglip(crops, texts):
    """Codifica recortes y textos con SigLIP. Devuelve matriz de similitud coseno."""
    image_t = torch.stack([preprocess(Image.fromarray(c)).to(device) for c in crops])
    text_t = tokenizer(texts).to(device)
    with torch.no_grad():
        img_emb = model.encode_image(image_t)
        txt_emb = model.encode_text(text_t)
    # Normaliza los embeddings antes de calcular la similitud (igual que arriba)
    img_emb = ... # <-- COMPLETE HERE
    txt_emb = ... # <-- COMPLETE HERE
    return cosine_similarity(txt_emb, img_emb)

def visualize_sam_siglip(img_np, similarity, binary_masks, text_queries, title='SAM + SigLIP'):
    """Superpone en verde la máscara con mayor similitud para cada query."""
    _, idx = torch.topk(similarity, dim=1, k=1)
    idx = idx.cpu()
    fig, axes = plt.subplots(1, len(text_queries), figsize=(5 * len(text_queries), 5))
    if len(text_queries) == 1:
        axes = [axes]
    for k in range(len(text_queries)):
        axes[k].imshow(img_np)
        axes[k].imshow(binary_masks[idx[k, 0].item()], cmap='Greens', alpha=0.5)
        axes[k].set_title(text_queries[k], fontweight='bold')
        axes[k].axis('off')
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Imagenes recortadas en: cropped_boxes_black
# Faltan los queries de texto

text_str = ["a bottle",
            "a computer monitor",
            "a game pad",
            "a laptop computer"]

image_embedding, text_embedding = ... # <-- COMPLETE HERE
similarity = ...  # <-- COMPLETE HERE


Visualizamos el resultado superponiendo la máscara ganadora en verde sobre la imagen original y una imagen por cada query de texto.

In [ ]:
def visualize_sam_siglip(img_np, similarity, binary_masks, text_queries, title='SAM + SigLIP'):
    _, idx = torch.topk(similarity, dim=1, k=1)
    idx = idx.cpu()
    fig, axes = plt.subplots(1, len(text_queries), figsize=(5 * len(text_queries), 5))
    if len(text_queries) == 1:
        axes = [axes]
    for k in range(len(text_queries)):
        axes[k].imshow(img_np)
        axes[k].imshow(binary_masks[idx[k, 0].item()], cmap='Greens', alpha=0.5)
        sim_score = similarity[k, idx[k, 0].item()].item()
        axes[k].set_title(f'{text_queries[k]}\n(sim: {sim_score:.3f})', fontweight='bold')
        axes[k].axis('off')
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_sam_siglip(img_src_np, similarity, batch_src_binary_mask, text_str)

# Prueba con tu propia imagen y tus propios prompts

Adelante, ahora es tu turno de jugar con el sistema creado. Ha llegado el momento de usar SAM + CLIP con una imagen tuya. Elige una imagen que tenga varios objetos distinguibles (una habitación, una mesa con cosas, una calle...) y escribe tus propias queries de texto.

Tienes dos opciones para cargar la imagen:
- **Opción A** — pega la URL de una imagen de internet
- **Opción B** — sube un archivo desde tu ordenador (solo funciona en Google Colab)

Luego define al menos **3 queries** describiendo objetos que crees que SAM + CLIP podrá encontrar. ¿Acierta? ¿En qué falla?

In [ ]:
from google.colab import files

# --- OPCIÓN A: URL ---
my_img_url = ... # <-- COMPLETE HERE

# --- OPCIÓN B: subir desde tu ordenador ---
# Descomenta las dos líneas siguientes para subir un archivo local:
# uploaded = files.upload()
# my_img_url = list(uploaded.keys())[0]  # nombre del archivo subido

# Cargamos la imagen
if my_img_url.startswith("http"):
    my_img, my_img_np = download_and_resize_image(my_img_url)
else:
    my_img = Image.open(my_img_url).convert("RGB").resize((1024, 768))
    my_img_np = np.array(my_img)

plt.imshow(my_img)
plt.axis("off")
plt.show()

### Escribe tus promts de texto que servirán como queries y obten las mascaras de SAM

In [ ]:
# Define tus propias queries — al menos 3 objetos que creas que están en la imagen
my_text_queries = [
    "a ...", # <-- COMPLETE HERE
    "a ...", # <-- COMPLETE HERE
    "a ...", # <-- COMPLETE HERE
]

# SAM genera las máscaras de tu imagen
my_sam_result = ... # <-- COMPLETE HERE
print(f"SAM encontró {len(my_sam_result)} máscaras")

## Recorta la imagen usando las mascaras de SAm y visualizamos los recortes


In [ ]:
# Recortamos y visualizamos las mascaras
my_crops, my_masks = ... # <-- COMPLETE HERE
visualize_cropped_masks(my_crops)

## Calcula los embeddings, el cosine similarity y visualiza los resultados.*italicized text*

In [ ]:
my_img_embb, my_txt_embb = ... # <-- COMPLETE HERE
my_similarity = ... # <-- COMPLETE HERE
visualize_sam_siglip(my_img_np, my_similarity, my_masks, my_text_queries, title="SAM + SigLIP: tu imagen")


# Preguntas de evaluación

**Responde a las siguientes preguntas en este notebook.**

---

### **Pregunta 1** 
**SAM no sabe qué objetos está recortando, solo los detecta visualmente. ¿Qué componente del pipeline es el que aporta la semántica para encontrar el objeto que buscas? Explica en una frase cómo lo hace.**

*Escribe tu respuesta aquí*

---

### **Pregunta 2**
**En este pipeline pasamos a SigLIP cada recorte de máscara por separado, no la imagen completa. ¿Por qué es necesario este paso? Piensa enqué ocurriría si pasaras la imagen entera: ¿qué embedding obtendrías y cómo afectaría a la comparación con el texto?**

*Escribe tu respuesta aquí*

---

### **Pregunta 3.**
**Este pipeline combina dos modelos que fueron entrenados de forma completamente independiente: SAM no sabe nada de lenguaje y SigLIP no fue entrenado para segmentar. Sin embargo, juntos resuelven un problema que ninguno de los dos podría resolver solo. ¿Qué ventajas tiene este enfoque de combinar modelos especializados frente a entrenar un único modelo que haga todo?**

*Escribe tu respuesta aquí*

---
